# 06 — Few-shot model and protocol comparison

This notebook compares the frozen DINOv2 baselines and the graph models under several few-shot protocols.

Default protocols:

- 5-way 5-shot
- 5-way 1-shot
- 10-way 5-shot
- 10-way 1-shot

Models:

- Frozen DINOv2 CLS prototype baseline
- Frozen DINOv2 mean-patch prototype baseline
- Pure GraphSAGE
- Residual GraphSAGE: `CLS + α × GraphSAGE`
- Pure GATv2
- Residual GATv2: `CLS + α × GATv2`

For every available model/protocol pair the notebook reports:

- cross-entropy loss
- query-level accuracy
- mean episode accuracy
- 95% CI across episode accuracies
- gain/loss versus frozen CLS
- paired episode-level gain versus CLS with 95% CI
- number of CLS errors fixed by the model
- number of CLS-correct predictions broken by the model
- fix rate and break rate
- accuracy as a function of frozen-CLS confidence margin
- learned residual scale `α` for residual models
- checkpoint epoch and validation accuracy
- evaluation runtime

## Important experimental rule

By default `MATCHED_TRAINING_ONLY = True`.

That means a checkpoint trained on 5-way 5-shot will **not** be presented as a normal 5-way 1-shot or 10-way 1-shot result. A graph model is evaluated only when its config says it was trained for the same `(n_way, k_shot)` protocol.

If you deliberately want a cross-way/cross-shot generalization experiment, set `MATCHED_TRAINING_ONLY = False`. Such results are labeled as cross-protocol results and should not be mixed with matched-training benchmark results.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source:", SRC_DIR)


In [ ]:
import json
import math
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from cross_image_glot.config import DEFAULT_PATHS
from cross_image_glot.storage import restore_feature_splits, atomic_json_save
from cross_image_glot.data import (
    MiniImageNetFeatureDataset,
    FewShotFeatureEpisodeDataset,
)
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder
from cross_image_glot.baselines import frozen_baseline_episode
from cross_image_glot.models import (
    PatchGraphSAGEEncoder,
    PatchGATv2Encoder,
    MeanPrototypeCosineReadout,
    CrossImageGraphMatcher,
    BaselinePreservingResidualMatcher,
)
from cross_image_glot.training import (
    evaluate_feature_episode,
    evaluate_residual_feature_episode,
)

paths = DEFAULT_PATHS
paths.ensure_directories()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local root:", paths.local_root)


## 1. Evaluation configuration

The final benchmark uses 600 fixed test episodes and 15 query images per class, matching the existing evaluation notebooks.

`QUICK_MODE=True` is useful for checking that the notebook works before committing to the full evaluation.

The graph experiments become much more expensive as `n_way` grows because each query creates one graph for every candidate class.


In [ ]:
QUICK_MODE = False

NUM_EPISODES = 20 if QUICK_MODE else 600
QUERIES_PER_CLASS = 15
TEST_SEED = 20_000
MAX_CACHED_SHARDS = 6

# Strict benchmark mode:
# True  -> only evaluate graph checkpoints trained with the same way/shot.
# False -> if a matched config is absent, optionally reuse the 5W5S checkpoint
#          and label the result as cross-protocol generalization.
MATCHED_TRAINING_ONLY = True

BASELINE_TEMPERATURE = 0.1
LOG_INTERVAL = 20 if not QUICK_MODE else 5

PROTOCOLS = [
    {"name": "5W5S",  "n_way": 5,  "k_shot": 5},
    {"name": "5W1S",  "n_way": 5,  "k_shot": 1},
    {"name": "10W5S", "n_way": 10, "k_shot": 5},
    {"name": "10W1S", "n_way": 10, "k_shot": 1},
]

# Existing 5W5S configs plus recommended filenames for future matched runs.
#
# If you name future configs differently, edit only this dictionary.
MODEL_CONFIGS = {
    "5W5S": {
        "Pure GraphSAGE": "configs/graphsage_5shot.json",
        "Residual GraphSAGE": "configs/residual_5shot.json",
        "Pure GATv2": "configs/gatv2_5shot.json",
        "Residual GATv2": "configs/residual_gatv2_5shot.json",
    },
    "5W1S": {
        "Pure GraphSAGE": "configs/graphsage_5way1shot.json",
        "Residual GraphSAGE": "configs/residual_graphsage_5way1shot.json",
        "Pure GATv2": "configs/gatv2_5way1shot.json",
        "Residual GATv2": "configs/residual_gatv2_5way1shot.json",
    },
    "10W5S": {
        "Pure GraphSAGE": "configs/graphsage_10way5shot.json",
        "Residual GraphSAGE": "configs/residual_graphsage_10way5shot.json",
        "Pure GATv2": "configs/gatv2_10way5shot.json",
        "Residual GATv2": "configs/residual_gatv2_10way5shot.json",
    },
    "10W1S": {
        "Pure GraphSAGE": "configs/graphsage_10way1shot.json",
        "Residual GraphSAGE": "configs/residual_graphsage_10way1shot.json",
        "Pure GATv2": "configs/gatv2_10way1shot.json",
        "Residual GATv2": "configs/residual_gatv2_10way1shot.json",
    },
}

# Used only if MATCHED_TRAINING_ONLY=False and a matched config is unavailable.
CROSS_PROTOCOL_FALLBACK_CONFIGS = MODEL_CONFIGS["5W5S"].copy()

print("Episodes per protocol:", NUM_EPISODES)
print("Matched-training-only:", MATCHED_TRAINING_ONLY)


## 2. Restore the held-out test feature cache

All protocols use the same class-disjoint miniImageNet test split and the same frozen DINOv2 cache. No DINOv2 forward pass is required.


In [ ]:
restore_feature_splits(
    ["test"],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

test_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "test",
    max_cached_shards=MAX_CACHED_SHARDS,
)

print("Test images:", len(test_features))
print("Test classes:", len(test_features.class_ids))
print("Feature metadata:", test_features.metadata)


## 3. Metric helpers

The 95% confidence interval is computed across episode accuracies:

`CI95 = 1.96 × std(episode_accuracy) / sqrt(number_of_episodes)`

For model-vs-CLS gain, the notebook also computes a **paired** episode-level CI using the accuracy difference on the same episode.

CLS confidence margin is measured in raw cosine-similarity units, not temperature-scaled logits:

`margin = best CLS cosine score - second-best CLS cosine score`

This makes the margin thresholds independent of the chosen softmax temperature.


In [ ]:
MARGIN_BINS = [-np.inf, 0.02, 0.05, 0.10, 0.20, np.inf]
MARGIN_LABELS = ["≤0.02", "0.02–0.05", "0.05–0.10", "0.10–0.20", ">0.20"]


def ci95(values):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return float("nan")
    return 1.96 * values.std(ddof=1) / math.sqrt(len(values))


def load_json(path):
    path = Path(path)
    return json.loads(path.read_text())


def graph_temperature(config):
    return float(
        config.get(
            "graph_temperature",
            config.get("temperature", 0.1),
        )
    )


def microbatch_size(config):
    return int(config.get("graph_microbatch_size", 2))


def summarize_from_episode_outputs(
    model_name,
    protocol_name,
    episode_losses,
    episode_accuracies,
    correct,
    num_queries,
    runtime_seconds,
    extra=None,
):
    summary = {
        "protocol": protocol_name,
        "model": model_name,
        "loss": float(np.mean(episode_losses)),
        "accuracy": correct / num_queries,
        "accuracy_percent": 100.0 * correct / num_queries,
        "correct": int(correct),
        "num_queries": int(num_queries),
        "num_episodes": int(len(episode_accuracies)),
        "episode_accuracy_mean": float(np.mean(episode_accuracies)),
        "episode_accuracy_ci95": float(ci95(episode_accuracies)),
        "episode_accuracy_ci95_pp": float(100.0 * ci95(episode_accuracies)),
        "runtime_seconds": float(runtime_seconds),
    }
    if extra:
        summary.update(extra)
    return summary


## 4. Detailed frozen-baseline evaluation

This function keeps per-query predictions so later cells can measure which CLS errors are fixed or broken by the graph models.


In [ ]:
@torch.inference_mode()
def evaluate_baseline_detailed(
    episode_dataset,
    representation,
    model_name,
    protocol_name,
    num_episodes,
    temperature=0.1,
):
    start = time.perf_counter()

    episode_losses = []
    episode_accuracies = []
    query_rows = []

    total_correct = 0
    total_queries = 0

    for episode_index in range(num_episodes):
        episode = episode_dataset[episode_index]

        logits, targets = frozen_baseline_episode(
            episode,
            representation,
            device,
            temperature=temperature,
        )

        loss = F.cross_entropy(logits, targets)
        predictions = logits.argmax(dim=-1)
        correct_mask = predictions.eq(targets)

        num_queries_episode = targets.numel()
        correct_episode = int(correct_mask.sum().item())

        episode_losses.append(float(loss.item()))
        episode_accuracies.append(correct_episode / num_queries_episode)
        total_correct += correct_episode
        total_queries += num_queries_episode

        if representation == "cls":
            # logits = cosine / temperature
            cosine_scores = logits * temperature
            top2 = cosine_scores.topk(k=2, dim=-1).values
            margins = top2[:, 0] - top2[:, 1]
        else:
            margins = torch.full(
                (num_queries_episode,),
                float("nan"),
                device=logits.device,
            )

        for query_index in range(num_queries_episode):
            query_rows.append(
                {
                    "protocol": protocol_name,
                    "episode_index": episode_index,
                    "query_index": query_index,
                    "target": int(targets[query_index].item()),
                    "prediction": int(predictions[query_index].item()),
                    "correct": bool(correct_mask[query_index].item()),
                    "cls_margin": float(margins[query_index].item()),
                }
            )

        if LOG_INTERVAL > 0 and (episode_index + 1) % LOG_INTERVAL == 0:
            print(
                f"  {protocol_name} {model_name}: "
                f"{episode_index + 1}/{num_episodes} episodes, "
                f"accuracy={total_correct / total_queries:.4f}"
            )

    summary = summarize_from_episode_outputs(
        model_name=model_name,
        protocol_name=protocol_name,
        episode_losses=episode_losses,
        episode_accuracies=episode_accuracies,
        correct=total_correct,
        num_queries=total_queries,
        runtime_seconds=time.perf_counter() - start,
        extra={"status": "evaluated", "training_protocol": "frozen"},
    )

    return {
        "summary": summary,
        "episode_accuracies": np.asarray(episode_accuracies, dtype=float),
        "queries": pd.DataFrame(query_rows),
    }


## 5. Model factory and checkpoint loader

The architecture is reconstructed from the same JSON used during training.

The notebook validates the training protocol stored in the config. In strict mode, a 5W5S checkpoint cannot silently become a 5W1S benchmark result.


In [ ]:
MODEL_KIND = {
    "Pure GraphSAGE": "graphsage",
    "Residual GraphSAGE": "residual_graphsage",
    "Pure GATv2": "gatv2",
    "Residual GATv2": "residual_gatv2",
}


def build_graph_matcher(kind, config):
    if "gatv2" in kind:
        encoder = PatchGATv2Encoder(
            input_dim=config["input_dim"],
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            heads=config["attention_heads"],
            edge_dim=config["edge_dim"],
            dropout=config["dropout"],
        )
    else:
        encoder = PatchGraphSAGEEncoder(
            input_dim=config["input_dim"],
            hidden_dim=config["hidden_dim"],
            num_layers=config["num_layers"],
            dropout=config["dropout"],
        )

    readout = MeanPrototypeCosineReadout(
        temperature=graph_temperature(config),
        learnable_temperature=False,
    )

    return CrossImageGraphMatcher(
        encoder=encoder,
        readout=readout,
    )


def build_and_load_model(model_name, config_path):
    config_path = Path(config_path)
    config = load_json(config_path)
    kind = MODEL_KIND[model_name]

    graph_matcher = build_graph_matcher(kind, config)

    if kind.startswith("residual_"):
        model = BaselinePreservingResidualMatcher(
            graph_matcher,
            config.get("initial_residual_scale", 0.0),
        )
    else:
        model = graph_matcher

    checkpoint_path = (
        paths.drive_checkpoint_dir
        / config["experiment_name"]
        / "best.pt"
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    return model, config, checkpoint, checkpoint_path


def resolve_model_config(protocol, model_name):
    protocol_name = protocol["name"]
    requested = Path(MODEL_CONFIGS[protocol_name][model_name])

    if requested.exists():
        config = load_json(requested)
        return requested, config, False

    if MATCHED_TRAINING_ONLY:
        return None, None, False

    fallback = Path(CROSS_PROTOCOL_FALLBACK_CONFIGS[model_name])
    if fallback.exists():
        config = load_json(fallback)
        return fallback, config, True

    return None, None, False


def validate_training_protocol(protocol, config, cross_protocol):
    trained = (int(config["n_way"]), int(config["k_shot"]))
    evaluated = (int(protocol["n_way"]), int(protocol["k_shot"]))

    if trained != evaluated and not cross_protocol:
        raise ValueError(
            f"Checkpoint config was trained on {trained}, "
            f"but evaluation protocol is {evaluated}."
        )

    return trained


## 6. Detailed graph-model evaluation

Every graph model is evaluated on exactly the same deterministic episode dataset as the baselines.

The per-query output is retained so we can compare each graph prediction directly against CLS.


In [ ]:
@torch.inference_mode()
def evaluate_graph_model_detailed(
    model,
    model_name,
    config,
    checkpoint,
    graph_builder,
    episode_dataset,
    protocol_name,
    num_episodes,
    cls_temperature=0.1,
):
    start = time.perf_counter()

    is_residual = MODEL_KIND[model_name].startswith("residual_")

    episode_losses = []
    episode_accuracies = []
    query_rows = []

    total_correct = 0
    total_queries = 0

    for episode_index in range(num_episodes):
        episode = episode_dataset[episode_index]

        if is_residual:
            result = evaluate_residual_feature_episode(
                model,
                graph_builder,
                episode,
                device,
                graph_microbatch_size=microbatch_size(config),
                cls_temperature=cls_temperature,
            )
        else:
            result = evaluate_feature_episode(
                model,
                graph_builder,
                episode,
                device,
                graph_microbatch_size=microbatch_size(config),
            )

        logits = result.logits.cpu()
        targets = result.targets.cpu().long()
        predictions = logits.argmax(dim=-1)
        correct_mask = predictions.eq(targets)

        # Recompute frozen CLS scores only for paired diagnostics.
        cls_logits, cls_targets = frozen_baseline_episode(
            episode,
            "cls",
            device,
            temperature=cls_temperature,
        )
        cls_logits = cls_logits.cpu()
        cls_targets = cls_targets.cpu().long()

        if not torch.equal(targets, cls_targets):
            raise RuntimeError(
                "Graph-model and CLS query order/targets do not match."
            )

        cls_predictions = cls_logits.argmax(dim=-1)
        cls_correct = cls_predictions.eq(targets)

        cls_cosine = cls_logits * cls_temperature
        top2 = cls_cosine.topk(k=2, dim=-1).values
        cls_margin = top2[:, 0] - top2[:, 1]

        episode_losses.append(float(result.loss))
        episode_accuracies.append(float(result.accuracy))
        total_correct += int(result.correct)
        total_queries += int(result.num_queries)

        for query_index in range(targets.numel()):
            query_rows.append(
                {
                    "protocol": protocol_name,
                    "episode_index": episode_index,
                    "query_index": query_index,
                    "target": int(targets[query_index].item()),
                    "prediction": int(predictions[query_index].item()),
                    "correct": bool(correct_mask[query_index].item()),
                    "cls_prediction": int(cls_predictions[query_index].item()),
                    "cls_correct": bool(cls_correct[query_index].item()),
                    "cls_margin": float(cls_margin[query_index].item()),
                }
            )

        if LOG_INTERVAL > 0 and (episode_index + 1) % LOG_INTERVAL == 0:
            print(
                f"  {protocol_name} {model_name}: "
                f"{episode_index + 1}/{num_episodes} episodes, "
                f"accuracy={total_correct / total_queries:.4f}"
            )

    extra = {
        "status": "evaluated",
        "checkpoint_epoch": checkpoint.get("epoch", None),
        "checkpoint_best_validation_accuracy": checkpoint.get(
            "best_validation_accuracy",
            None,
        ),
    }

    if is_residual:
        extra["residual_scale"] = float(
            model.residual_scale.detach().cpu().item()
        )
    else:
        extra["residual_scale"] = float("nan")

    summary = summarize_from_episode_outputs(
        model_name=model_name,
        protocol_name=protocol_name,
        episode_losses=episode_losses,
        episode_accuracies=episode_accuracies,
        correct=total_correct,
        num_queries=total_queries,
        runtime_seconds=time.perf_counter() - start,
        extra=extra,
    )

    return {
        "summary": summary,
        "episode_accuracies": np.asarray(episode_accuracies, dtype=float),
        "queries": pd.DataFrame(query_rows),
    }


## 7. Paired comparison against CLS

`fixed_by_model` counts cases where CLS was wrong and the compared model became correct.

`broken_by_model` counts cases where CLS was correct and the compared model became wrong.

A useful net paired count is:

`net_fixed = fixed_by_model - broken_by_model`

The notebook also reports the fraction of all CLS errors fixed and the fraction of all CLS-correct decisions broken.


In [ ]:
def add_cls_comparison(model_result, cls_result):
    summary = dict(model_result["summary"])

    cls_summary = cls_result["summary"]

    summary["delta_vs_cls_pp"] = (
        summary["accuracy_percent"]
        - cls_summary["accuracy_percent"]
    )

    model_ep = np.asarray(model_result["episode_accuracies"])
    cls_ep = np.asarray(cls_result["episode_accuracies"])

    if len(model_ep) != len(cls_ep):
        raise RuntimeError(
            "Model and CLS episode counts do not match."
        )

    paired_delta = model_ep - cls_ep

    summary["paired_episode_gain_mean_pp"] = (
        100.0 * float(paired_delta.mean())
    )
    summary["paired_episode_gain_ci95_pp"] = (
        100.0 * float(ci95(paired_delta))
    )

    model_q = model_result["queries"].copy()
    cls_q = cls_result["queries"][
        [
            "episode_index",
            "query_index",
            "target",
            "prediction",
            "correct",
            "cls_margin",
        ]
    ].copy()

    cls_q = cls_q.rename(
        columns={
            "prediction": "cls_prediction_reference",
            "correct": "cls_correct_reference",
            "cls_margin": "cls_margin_reference",
        }
    )

    merged = model_q.merge(
        cls_q,
        on=["episode_index", "query_index", "target"],
        how="inner",
        validate="one_to_one",
    )

    model_correct = merged["correct"].astype(bool)
    cls_correct = merged["cls_correct_reference"].astype(bool)

    fixed = int((~cls_correct & model_correct).sum())
    broken = int((cls_correct & ~model_correct).sum())

    cls_errors = int((~cls_correct).sum())
    cls_correct_count = int(cls_correct.sum())

    summary["fixed_by_model"] = fixed
    summary["broken_by_model"] = broken
    summary["net_fixed"] = fixed - broken
    summary["fix_rate_of_cls_errors"] = (
        fixed / cls_errors if cls_errors else float("nan")
    )
    summary["break_rate_of_cls_correct"] = (
        broken / cls_correct_count
        if cls_correct_count
        else float("nan")
    )

    return summary


def margin_analysis(model_result, cls_result, model_name, protocol_name):
    model_q = model_result["queries"][
        ["episode_index", "query_index", "target", "correct"]
    ].copy()

    cls_q = cls_result["queries"][
        [
            "episode_index",
            "query_index",
            "target",
            "correct",
            "cls_margin",
        ]
    ].copy()

    model_q = model_q.rename(columns={"correct": "model_correct"})
    cls_q = cls_q.rename(columns={"correct": "cls_correct"})

    merged = model_q.merge(
        cls_q,
        on=["episode_index", "query_index", "target"],
        validate="one_to_one",
    )

    merged["margin_bin"] = pd.cut(
        merged["cls_margin"],
        bins=MARGIN_BINS,
        labels=MARGIN_LABELS,
        include_lowest=True,
        right=True,
    )

    rows = []

    for margin_bin in MARGIN_LABELS:
        subset = merged[merged["margin_bin"] == margin_bin]

        if len(subset) == 0:
            continue

        cls_acc = subset["cls_correct"].mean()
        model_acc = subset["model_correct"].mean()

        fixed = (
            (~subset["cls_correct"])
            & subset["model_correct"]
        ).sum()

        broken = (
            subset["cls_correct"]
            & (~subset["model_correct"])
        ).sum()

        rows.append(
            {
                "protocol": protocol_name,
                "model": model_name,
                "margin_bin": margin_bin,
                "num_queries": len(subset),
                "cls_accuracy": float(cls_acc),
                "model_accuracy": float(model_acc),
                "gain_vs_cls_pp": float(
                    100.0 * (model_acc - cls_acc)
                ),
                "fixed_by_model": int(fixed),
                "broken_by_model": int(broken),
            }
        )

    return rows


## 8. Run the protocol comparison

Baselines are always evaluated.

For graph models:

- matched config + checkpoint → evaluated;
- missing matched config/checkpoint → skipped and recorded;
- if `MATCHED_TRAINING_ONLY=False`, the notebook may use the 5W5S checkpoint as a deliberately labeled cross-protocol generalization test.

The graph builder uses the `top_k` stored in the model's own training config.


In [ ]:
all_summary_rows = []
all_margin_rows = []
all_query_rows = []
skipped_rows = []

protocol_results = {}

for protocol in PROTOCOLS:
    protocol_name = protocol["name"]

    print("\n" + "=" * 80)
    print(
        f"{protocol_name}: "
        f"{protocol['n_way']}-way "
        f"{protocol['k_shot']}-shot"
    )
    print("=" * 80)

    episodes = FewShotFeatureEpisodeDataset(
        test_features,
        n_way=protocol["n_way"],
        k_shot=protocol["k_shot"],
        queries_per_class=QUERIES_PER_CLASS,
        num_episodes=NUM_EPISODES,
        seed=TEST_SEED,
        vary_by_epoch=False,
    )

    # -------- Frozen CLS baseline --------
    cls_result = evaluate_baseline_detailed(
        episodes,
        representation="cls",
        model_name="Frozen CLS",
        protocol_name=protocol_name,
        num_episodes=NUM_EPISODES,
        temperature=BASELINE_TEMPERATURE,
    )

    cls_summary = dict(cls_result["summary"])
    cls_summary.update(
        {
            "delta_vs_cls_pp": 0.0,
            "paired_episode_gain_mean_pp": 0.0,
            "paired_episode_gain_ci95_pp": 0.0,
            "fixed_by_model": 0,
            "broken_by_model": 0,
            "net_fixed": 0,
            "fix_rate_of_cls_errors": 0.0,
            "break_rate_of_cls_correct": 0.0,
            "residual_scale": float("nan"),
        }
    )
    all_summary_rows.append(cls_summary)

    cls_queries_to_save = cls_result["queries"].copy()
    cls_queries_to_save["model"] = "Frozen CLS"
    all_query_rows.append(cls_queries_to_save)

    # -------- Frozen mean-patch baseline --------
    mean_result = evaluate_baseline_detailed(
        episodes,
        representation="mean_patch",
        model_name="Frozen mean-patch",
        protocol_name=protocol_name,
        num_episodes=NUM_EPISODES,
        temperature=BASELINE_TEMPERATURE,
    )

    mean_summary = add_cls_comparison(
        mean_result,
        cls_result,
    )
    all_summary_rows.append(mean_summary)

    mean_queries_to_save = mean_result["queries"].copy()
    mean_queries_to_save["model"] = "Frozen mean-patch"
    all_query_rows.append(mean_queries_to_save)

    all_margin_rows.extend(
        margin_analysis(
            mean_result,
            cls_result,
            "Frozen mean-patch",
            protocol_name,
        )
    )

    protocol_results[protocol_name] = {
        "Frozen CLS": cls_result,
        "Frozen mean-patch": mean_result,
    }

    # -------- Graph models --------
    for model_name in MODEL_KIND:
        resolved_path, candidate_config, cross_protocol = (
            resolve_model_config(protocol, model_name)
        )

        if resolved_path is None:
            reason = "matched config not found"
            print(f"SKIP {model_name}: {reason}")
            skipped_rows.append(
                {
                    "protocol": protocol_name,
                    "model": model_name,
                    "reason": reason,
                }
            )
            continue

        trained_protocol = validate_training_protocol(
            protocol,
            candidate_config,
            cross_protocol,
        )

        checkpoint_path = (
            paths.drive_checkpoint_dir
            / candidate_config["experiment_name"]
            / "best.pt"
        )

        if not checkpoint_path.exists():
            reason = f"checkpoint not found: {checkpoint_path}"
            print(f"SKIP {model_name}: {reason}")
            skipped_rows.append(
                {
                    "protocol": protocol_name,
                    "model": model_name,
                    "reason": reason,
                }
            )
            continue

        print(
            f"\nEvaluating {model_name} "
            f"(trained {trained_protocol[0]}W{trained_protocol[1]}S)"
        )

        model, model_config, checkpoint, checkpoint_path = (
            build_and_load_model(
                model_name,
                resolved_path,
            )
        )

        graph_builder = ClassConditionedPatchGraphBuilder(
            grid_size=tuple(
                test_features.metadata["grid_size"]
            ),
            top_k=model_config["top_k"],
            min_similarity=model_config.get(
                "min_similarity",
                None,
            ),
            graph_dtype=torch.float32,
            similarity_device=device,
        )

        result = evaluate_graph_model_detailed(
            model=model,
            model_name=model_name,
            config=model_config,
            checkpoint=checkpoint,
            graph_builder=graph_builder,
            episode_dataset=episodes,
            protocol_name=protocol_name,
            num_episodes=NUM_EPISODES,
            cls_temperature=float(
                model_config.get(
                    "cls_temperature",
                    BASELINE_TEMPERATURE,
                )
            ),
        )

        result["summary"]["training_protocol"] = (
            f"{trained_protocol[0]}W{trained_protocol[1]}S"
        )
        result["summary"]["cross_protocol_generalization"] = (
            bool(cross_protocol)
        )
        result["summary"]["checkpoint_path"] = str(
            checkpoint_path
        )

        compared_summary = add_cls_comparison(
            result,
            cls_result,
        )

        all_summary_rows.append(compared_summary)

        model_queries_to_save = result["queries"].copy()
        model_queries_to_save["model"] = model_name
        all_query_rows.append(model_queries_to_save)

        all_margin_rows.extend(
            margin_analysis(
                result,
                cls_result,
                model_name,
                protocol_name,
            )
        )

        protocol_results[protocol_name][model_name] = result

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

summary_df = pd.DataFrame(all_summary_rows)
margin_df = pd.DataFrame(all_margin_rows)
skipped_df = pd.DataFrame(skipped_rows)

query_df = (
    pd.concat(all_query_rows, ignore_index=True)
    if all_query_rows
    else pd.DataFrame()
)

print("\nFinished.")


## 9. Main comparison table

`delta_vs_cls_pp` is the absolute percentage-point change versus the frozen CLS prototype classifier.

`paired_episode_gain_ci95_pp` is the 95% CI of the **paired episode-level improvement**, which is more informative than comparing two independent confidence intervals.

Positive `net_fixed` means the model corrected more CLS decisions than it damaged.


In [ ]:
display_columns = [
    "protocol",
    "model",
    "loss",
    "accuracy_percent",
    "episode_accuracy_ci95_pp",
    "delta_vs_cls_pp",
    "paired_episode_gain_mean_pp",
    "paired_episode_gain_ci95_pp",
    "fixed_by_model",
    "broken_by_model",
    "net_fixed",
    "fix_rate_of_cls_errors",
    "break_rate_of_cls_correct",
    "residual_scale",
    "runtime_seconds",
    "training_protocol",
]

available_columns = [
    column
    for column in display_columns
    if column in summary_df.columns
]

comparison_table = (
    summary_df[available_columns]
    .sort_values(["protocol", "accuracy_percent"], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option("display.max_columns", None)
display(comparison_table)

if len(skipped_df):
    print("\nSkipped graph runs:")
    display(skipped_df)


## 10. Accuracy with 95% episode CI


In [ ]:
if len(summary_df):
    protocol_order = [p["name"] for p in PROTOCOLS]
    model_order = [
        "Frozen CLS",
        "Frozen mean-patch",
        "Pure GraphSAGE",
        "Residual GraphSAGE",
        "Pure GATv2",
        "Residual GATv2",
    ]

    fig, ax = plt.subplots(figsize=(13, 6))

    x = np.arange(len(protocol_order), dtype=float)
    present_models = [
        m
        for m in model_order
        if m in set(summary_df["model"])
    ]

    width = 0.8 / max(len(present_models), 1)

    for model_index, model_name in enumerate(present_models):
        values = []
        errors = []

        for protocol_name in protocol_order:
            row = summary_df[
                (summary_df["protocol"] == protocol_name)
                & (summary_df["model"] == model_name)
            ]

            if len(row):
                values.append(float(row.iloc[0]["accuracy_percent"]))
                errors.append(float(row.iloc[0]["episode_accuracy_ci95_pp"]))
            else:
                values.append(np.nan)
                errors.append(np.nan)

        offset = (
            model_index - (len(present_models) - 1) / 2
        ) * width

        ax.bar(
            x + offset,
            values,
            width=width,
            label=model_name,
            yerr=errors,
            capsize=3,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(protocol_order)
    ax.set_ylabel("Accuracy (%)")
    ax.set_xlabel("Few-shot protocol")
    ax.set_title("Few-shot accuracy with 95% episode confidence intervals")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.show()


## 11. Gain over frozen CLS


In [ ]:
non_cls = summary_df[summary_df["model"] != "Frozen CLS"].copy()

if len(non_cls):
    fig, ax = plt.subplots(figsize=(13, 6))

    for model_name, group in non_cls.groupby("model"):
        ordered = (
            group.set_index("protocol")
            .reindex([p["name"] for p in PROTOCOLS])
        )

        ax.plot(
            ordered.index,
            ordered["delta_vs_cls_pp"],
            marker="o",
            label=model_name,
        )

    ax.axhline(0.0, linewidth=1)
    ax.set_ylabel("Accuracy gain over frozen CLS (percentage points)")
    ax.set_xlabel("Few-shot protocol")
    ax.set_title("Model improvement relative to frozen DINOv2 CLS")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()


## 12. Performance versus frozen-CLS confidence margin

This analysis answers a central project question:

> Does graph reasoning help especially when the frozen global DINOv2 representation is uncertain?

Small CLS margin means the top two candidate classes received very similar CLS cosine scores.


In [ ]:
if len(margin_df):
    display(
        margin_df.sort_values(
            ["protocol", "model", "margin_bin"]
        ).reset_index(drop=True)
    )

    for protocol_name in margin_df["protocol"].unique():
        subset = margin_df[
            margin_df["protocol"] == protocol_name
        ]

        fig, ax = plt.subplots(figsize=(11, 5))

        for model_name, group in subset.groupby("model"):
            ordered = (
                group.set_index("margin_bin")
                .reindex(MARGIN_LABELS)
            )

            ax.plot(
                ordered.index,
                ordered["gain_vs_cls_pp"],
                marker="o",
                label=model_name,
            )

        ax.axhline(0.0, linewidth=1)
        ax.set_ylabel("Gain over CLS (percentage points)")
        ax.set_xlabel("Frozen CLS top-1 vs top-2 cosine margin")
        ax.set_title(
            f"{protocol_name}: gain as a function of CLS confidence"
        )
        ax.legend()
        ax.grid(alpha=0.25)
        plt.show()


## 13. Correction / damage analysis

The most useful columns are:

- `fixed_by_model`: CLS wrong → model correct
- `broken_by_model`: CLS correct → model wrong
- `net_fixed`: fixed - broken
- `fix_rate_of_cls_errors`: fraction of all CLS errors recovered
- `break_rate_of_cls_correct`: fraction of correct CLS predictions damaged


In [ ]:
correction_columns = [
    "protocol",
    "model",
    "fixed_by_model",
    "broken_by_model",
    "net_fixed",
    "fix_rate_of_cls_errors",
    "break_rate_of_cls_correct",
    "delta_vs_cls_pp",
]

display(
    summary_df[
        [c for c in correction_columns if c in summary_df.columns]
    ]
    .sort_values(["protocol", "net_fixed"], ascending=[True, False])
    .reset_index(drop=True)
)


## 14. Residual-scale analysis

For the residual models:

`final_logits = CLS_logits + α × graph_logits`

A residual scale close to zero means the trained model learned to rely mostly on CLS. A substantial nonzero value means the graph branch materially contributes to the final score.


In [ ]:
if "residual_scale" in summary_df.columns:
    residual_table = summary_df[
        summary_df["model"].isin(
            ["Residual GraphSAGE", "Residual GATv2"]
        )
    ][
        [
            "protocol",
            "model",
            "residual_scale",
            "accuracy_percent",
            "delta_vs_cls_pp",
            "fixed_by_model",
            "broken_by_model",
        ]
    ].reset_index(drop=True)

    display(residual_table)


## 15. Save all results to Google Drive

The notebook saves both aggregate and diagnostic outputs:

- `summary.csv`
- `margin_analysis.csv`
- `skipped_runs.csv`
- `query_predictions.csv`
- `comparison.json`

The query-level CSV is useful for later inspecting specific examples where GAT/GraphSAGE fixed or broke a CLS decision.


In [ ]:
output_dir = (
    paths.drive_results_dir
    / "fewshot_protocol_comparison"
)
output_dir.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(
    output_dir / "summary.csv",
    index=False,
)

margin_df.to_csv(
    output_dir / "margin_analysis.csv",
    index=False,
)

skipped_df.to_csv(
    output_dir / "skipped_runs.csv",
    index=False,
)

query_df.to_csv(
    output_dir / "query_predictions.csv",
    index=False,
)

json_payload = {
    "settings": {
        "num_episodes": NUM_EPISODES,
        "queries_per_class": QUERIES_PER_CLASS,
        "test_seed": TEST_SEED,
        "matched_training_only": MATCHED_TRAINING_ONLY,
        "baseline_temperature": BASELINE_TEMPERATURE,
        "protocols": PROTOCOLS,
    },
    "summary": summary_df.replace(
        {np.nan: None}
    ).to_dict(orient="records"),
    "margin_analysis": margin_df.replace(
        {np.nan: None}
    ).to_dict(orient="records"),
    "skipped_runs": skipped_df.to_dict(
        orient="records"
    ),
}

atomic_json_save(
    json_payload,
    output_dir / "comparison.json",
)

print("Saved comparison outputs to:")
print(output_dir)


## Interpretation guide

The strongest evidence for the project would not merely be a higher aggregate GAT accuracy.

A more informative pattern is:

1. frozen CLS remains a strong global baseline;
2. pure GATv2 improves substantially over frozen mean-patch pooling;
3. residual GATv2 improves over frozen CLS;
4. GATv2 fixes more CLS errors than it breaks;
5. the gain becomes larger in low-CLS-margin queries;
6. the gain increases as the few-shot task becomes harder, especially 1-shot and/or higher-way episodes.

That pattern would support the claim that cross-image patch reasoning contributes complementary local information rather than merely replacing a strong global DINOv2 representation.
